In [4]:
import multiprocessing
import os
import pickle
import sys
import gc
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import polars as pl

In [24]:
DEBUG = 1

In [25]:
ROOT = Path.cwd().parent.parent.parent

train_dir = ROOT / "data/raw/validation/train_parquet"
test_dir = ROOT / "data/raw/validation/test_parquet"

train_files = list(sorted(train_dir.glob("*.parquet")))
test_files = list(sorted(test_dir.glob("*.parquet")))

print(type(train_dir))
print(type(train_files))
print(len(train_files))
print(train_files[:3])

for path in train_files:
    bad = (
        pl.scan_parquet(str(path))
        .with_row_index("row_nr")
        .with_columns(
            prev_session=pl.col("session").shift(),
            prev_ts=pl.col("ts").shift(),
        )
        .filter(pl.col("row_nr") > 0)
        .filter(
            (pl.col("session") < pl.col("prev_session")) |
            (
                (pl.col("session") == pl.col("prev_session")) &
                (pl.col("ts") < pl.col("prev_ts"))
            )
        )
        .select("row_nr", "prev_session", "session", "prev_ts", "ts")
        .head(20)
        .collect()
    )

    print(path.name, end=" ")
    if bad.height == 0:
        print("已按 session, ts 升序排好序")
    else:
        print("没有排好序，前 20 条违规记录：")
        print(bad)

<class 'pathlib.PosixPath'>
<class 'list'>
100
[PosixPath('/Users/xiafan/Desktop/otto_self/data/raw/validation/train_parquet/000.parquet'), PosixPath('/Users/xiafan/Desktop/otto_self/data/raw/validation/train_parquet/001.parquet'), PosixPath('/Users/xiafan/Desktop/otto_self/data/raw/validation/train_parquet/002.parquet')]
000.parquet 已按 session, ts 升序排好序
001.parquet 已按 session, ts 升序排好序
002.parquet 已按 session, ts 升序排好序
003.parquet 已按 session, ts 升序排好序
004.parquet 已按 session, ts 升序排好序
005.parquet 已按 session, ts 升序排好序
006.parquet 已按 session, ts 升序排好序
007.parquet 已按 session, ts 升序排好序
008.parquet 已按 session, ts 升序排好序
009.parquet 已按 session, ts 升序排好序
010.parquet 已按 session, ts 升序排好序
011.parquet 已按 session, ts 升序排好序
012.parquet 已按 session, ts 升序排好序
013.parquet 已按 session, ts 升序排好序
014.parquet 已按 session, ts 升序排好序
015.parquet 已按 session, ts 升序排好序
016.parquet 已按 session, ts 升序排好序
017.parquet 已按 session, ts 升序排好序
018.parquet 已按 session, ts 升序排好序
019.parquet 已按 session, ts 升序排好序
020.parquet 已按 s

In [26]:
if DEBUG == True:
    train_files = train_files[0]

In [ ]:
class